# 20 — Environment device policy benchmark

Этот notebook нужен, чтобы проверить, где быстрее держать CrafText env rollout: CPU или GPU, и нужен ли `jit(vmap(step))`.

Ключевая деталь: `JAX_PLATFORMS` должен быть выставлен **до импорта JAX**. Если меняете CPU/GPU режим — поменяйте первую ячейку и сделайте `Restart Kernel`.


In [ ]:
import os

# Choose one before importing JAX. Restart kernel after changing it.
# CPU env + vLLM GPU in the same process usually wants JAX_PLATFORMS='cpu'.
os.environ.setdefault('JAX_PLATFORMS', 'cpu')
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

print('JAX_PLATFORMS=', os.environ.get('JAX_PLATFORMS'))
print('XLA_PYTHON_CLIENT_PREALLOCATE=', os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'])
print('VLLM_WORKER_MULTIPROC_METHOD=', os.environ['VLLM_WORKER_MULTIPROC_METHOD'])


In [ ]:
from pathlib import Path

import jax

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository')

CONFIG_PATH = ROOT / 'configs/env/text/qwen_craftext.yaml'
OUTPUT_PATH = ROOT / 'artifacts/benchmarks/env-device-policy-latest.json'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print('repo:', ROOT)
print('jax backend:', jax.default_backend())
print('jax devices:', jax.devices())


In [ ]:
from tunix_craftext.env.config import load_mvp_config
from tunix_craftext.env.prompts import MegaPromptRenderer
from tunix_craftext.env.runtime import build_craftext_runtime
from tunix_craftext.env.tasks import CrafTextTaskSampler
from tunix_craftext.models.llm import ScriptedLlmBackend

config = load_mvp_config(CONFIG_PATH)
runtime = build_craftext_runtime(config)
renderer = MegaPromptRenderer(config.prompt.template)

task_sampler = CrafTextTaskSampler.from_runtime(
    runtime,
    horizon=config.environment.horizon,
    mode='fixed',
    fixed_instruction_index=config.environment.instruction_index,
    goal_prefix=config.run.goal,
)
prepared_goal, prepared_instruction_index = task_sampler.task_at(config.environment.instruction_index)
fallback_action_id = runtime.actions.index_of('NOOP')
ACTION_LABEL = 'NOOP'


def backend_for_action(action_label: str) -> ScriptedLlmBackend:
    return ScriptedLlmBackend(f'<action>{action_label}</action>', model='env-device-policy')
print('instruction_index:', prepared_instruction_index)
print('goal:', prepared_goal)
print('actions:', runtime.actions.labels)
print('initial ACTION_LABEL:', ACTION_LABEL)


## Action terminal sweep

Если scripted action завершает все среды на каждом шаге, benchmark превращается в reset storm. Эта ячейка быстро проверяет несколько action labels и выбирает первый, где `finished_count < batch_size`. Если все проверенные actions terminal, расширьте `ACTION_SWEEP_LABELS` вручную.


In [ ]:
from tunix_craftext.rollouts.batched import (
    EnvironmentDevicePolicy,
    HostBatchPolicy,
    collect_batched_text_rollout_profiled,
    cpu_environment_device_policy,
)

ACTION_SWEEP_BATCH_SIZE = 4
ACTION_SWEEP_LABELS = tuple(runtime.actions.labels[: min(8, len(runtime.actions.labels))])


def summarize_action_label(action_label: str) -> dict[str, object]:
    profiled = collect_batched_text_rollout_profiled(
        runtime.adapter,
        renderer,
        backend_for_action(action_label),
        actions=runtime.actions,
        batch_size=ACTION_SWEEP_BATCH_SIZE,
        horizon=1,
        seed=config.run.seed,
        goal=prepared_goal,
        max_new_tokens=8,
        invalid_action='fallback',
        fallback_action_id=fallback_action_id,
        host_batch_policy=HostBatchPolicy(prompt_workers=4),
        device_policy=EnvironmentDevicePolicy(
            backend=None,
            jit_reset=True,
            jit_step=True,
            place_inputs=False,
            snapshot_prompt_inputs=True,
        ),
    )
    decision = profiled.rollout.decisions[0]
    return {
        'label': action_label,
        'events': profiled.event_totals(),
        'reward_sum': float(decision.transition.reward.sum()),
        'phase_totals_ms': profiled.phase_totals_ms(),
    }


action_summaries = [summarize_action_label(label) for label in ACTION_SWEEP_LABELS]
for summary in action_summaries:
    print(summary['label'], summary['events'], 'reward_sum=', summary['reward_sum'])

nonterminal_candidates = [
    summary for summary in action_summaries
    if summary['events']['finished_count'] < ACTION_SWEEP_BATCH_SIZE
]
if nonterminal_candidates:
    ACTION_LABEL = str(nonterminal_candidates[0]['label'])
else:
    print('All swept actions finished every row; edit ACTION_SWEEP_LABELS or accept reset-storm benchmark.')
print('selected ACTION_LABEL:', ACTION_LABEL)


## Modes

- `vmap_only`: `device_policy=None`; no explicit placement and no `jit` wrapper from the rollout collector.
- `jit_vmap_default_backend`: no explicit device placement, but reset/step are wrapped as `jit(vmap(...))`.
- `explicit_default_backend`: explicitly places env carry on `jax.default_backend()` and uses `jit(vmap(...))`.

Для проверки CPU env рядом с vLLM GPU используйте первую ячейку с `JAX_PLATFORMS='cpu'`. Для проверки GPU env перезапустите kernel без `JAX_PLATFORMS='cpu'` или с GPU-visible JAX install и сравните отчёты.


In [ ]:
from statistics import median

BATCH_SIZE = 8
HORIZON = 4
REPEATS = 3

modes = {
    'vmap_only': None,
    'jit_vmap_default_backend': EnvironmentDevicePolicy(
        backend=None,
        jit_reset=True,
        jit_step=True,
        place_inputs=False,
        snapshot_prompt_inputs=True,
    ),
    'explicit_default_backend': EnvironmentDevicePolicy(
        backend=jax.default_backend(),
        jit_reset=True,
        jit_step=True,
        place_inputs=True,
        snapshot_prompt_inputs=True,
    ),
    'explicit_cpu_sidecar': cpu_environment_device_policy(),
}


def run_profile(label: str, policy: EnvironmentDevicePolicy | None):
    profiled = collect_batched_text_rollout_profiled(
        runtime.adapter,
        renderer,
        backend_for_action(ACTION_LABEL),
        actions=runtime.actions,
        batch_size=BATCH_SIZE,
        horizon=HORIZON,
        seed=config.run.seed,
        goal=prepared_goal,
        max_new_tokens=8,
        invalid_action='fallback',
        fallback_action_id=fallback_action_id,
        host_batch_policy=HostBatchPolicy(prompt_workers=4),
        device_policy=policy,
    )
    return {
        'label': label,
        'phase_totals_ms': profiled.phase_totals_ms(),
        'event_totals': profiled.event_totals(),
        'per_step': [
            {
                'environment_step_ms': timing.decision.environment_step_ms,
                'reset_ms': timing.reset_ms,
                'replace_finished_ms': timing.replace_finished_ms,
                'finished_count': timing.finished_count,
                'reset_invoked': timing.reset_invoked,
                'total_ms': timing.total_ms,
            }
            for timing in profiled.timings
        ],
    }


print('warmup each mode once')
warmup = [run_profile(label, policy) for label, policy in modes.items()]
for item in warmup:
    print('warmup', item['label'], item['phase_totals_ms'])

runs = []
for repeat in range(REPEATS):
    labels = tuple(modes) if repeat % 2 == 0 else tuple(reversed(tuple(modes)))
    for label in labels:
        result = run_profile(label, modes[label])
        result['repeat'] = repeat
        runs.append(result)
        print('run', repeat, label, result['phase_totals_ms'])


In [ ]:
def median_totals(label: str) -> dict[str, float]:
    label_runs = [run for run in runs if run['label'] == label]
    phases = label_runs[0]['phase_totals_ms']
    return {phase: median(run['phase_totals_ms'][phase] for run in label_runs) for phase in phases}


summary = {label: median_totals(label) for label in modes}
phases = tuple(next(iter(summary.values())).keys())

print('Median phase totals after warmup')
print(f"{'phase':28s} " + ' '.join(f'{label:>28s}' for label in summary))
print('-' * (29 + 29 * len(summary)))
for phase in phases:
    print(f"{phase:28s} " + ' '.join(f"{summary[label][phase]:28.2f}" for label in summary))

print('\nEvent totals from last run per mode')
for label in modes:
    last = [run for run in runs if run['label'] == label][-1]
    print(label, last['event_totals'])


In [ ]:
import json

report = {
    'config_path': str(CONFIG_PATH.relative_to(ROOT)),
    'jax_platforms': os.environ.get('JAX_PLATFORMS'),
    'jax_backend': jax.default_backend(),
    'jax_devices': [str(device) for device in jax.devices()],
    'batch_size': BATCH_SIZE,
    'horizon': HORIZON,
    'repeats': REPEATS,
    'action_label': ACTION_LABEL,
    'action_summaries': action_summaries,
    'summary': summary,
    'warmup': warmup,
    'runs': runs,
}
OUTPUT_PATH.write_text(json.dumps(report, indent=2, sort_keys=True), encoding='utf-8')
print('saved:', OUTPUT_PATH)
